# ray start
ray start 是 Ray 分布式框架中最基础的命令行工具，用于在物理机或虚拟机上启动 Ray 节点进程。它是构建 Ray 集群的“积木”。
通过组合不同的参数，你可以用它来搭建从“单机测试”到“大规模多机集群”的各种环境。
以下是 ray start 的详细介绍，包括核心参数、常用场景和底层原理。

## 🏗️ 核心架构与命令模式

Ray 集群采用 Master-Slave 架构，因此 ray start 命令主要对应两种启动模式：
- <font color='red'>Head Node (主节点)：</font>集群的大脑。负责管理集群状态、调度任务、监控资源。
- <font color='red'>Worker Node (工作节点)：</font>集群的苦力。负责执行具体的计算任务（Task）和Actor。

## 🔑 常用参数详解

| 参数 | 说明 | 默认值 | 备注 |
| :--- | :--- | :--- | :--- |
| `--head` | 指定当前节点为 Head Node。 | 无 | 如果不加此参数，默认启动的是 Worker Node。 |
| `--address` | 指定要连接的 Head Node 地址。 | 无 | Worker 节点启动时必须指定，格式为 `IP:Port`。 |
| `--port` | 指定 Head Node 的通信端口（Redis端口）。 | `6379` | 如果默认端口被占用，需手动指定。 |
| `--num-cpus` | <font color='red'>指定 Ray 可见的 CPU 核心数。 </font>| 自动检测 | 用于限制 Ray 在该节点上能调度的 CPU 资源。 |
| `--num-gpus` | <font color='red'>指定 Ray 可见的 GPU 数量。 </font>| 自动检测 | 用于限制 Ray 在该节点上能调度的 GPU 资源。 |
| `--include-dashboard` | 是否启动 Web 可视化监控界面。 | `True` | <font color='red'>仅在 Head Node 上有效。</font> |
| `--dashboard-host` | 指定 Dashboard 的监听地址。 | `localhost` | 若要远程访问，需设为 `0.0.0.0`。 |
| `--dashboard-port` | 指定 Dashboard 的端口。 | `8265` | 若端口冲突可修改。 |
| `--node-ip-address` | 指定当前节点的 IP 地址。 | 自动检测 | <font color='red'>在多网卡环境下，需手动指定用于集群通信的 IP。 </font>|


## 🚀 常用场景与实操示例
### 场景一：本地单机开发（最常用）

这是开发者最常用的模式，用于在本地笔记本或服务器上快速测试代码。

命令：

In [ ]:
ray start --head

- 解释：在当前机器启动一个 Head Node，同时它也是一个 Worker。Ray 会自动检测 CPU/GPU，并启动 Dashboard。
- 验证：在 Python 中运行 import ray; ray.init() 即可连接。

### 场景二：搭建多机集群（生产环境标准做法）
假设你有两台机器：
- Head 节点 IP: 192.168.1.100
- Worker 节点 IP: 192.168.1.101
#### 步骤 1：在 Head 节点启动

In [ ]:
ray start --head --port=6379 --include-dashboard=true --dashboard-host=0.0.0.0

- 注意：--dashboard-host=0.0.0.0 允许你通过浏览器访问 Head 节点的 IP 来查看监控界面。
- 输出：命令执行成功后，终端会打印类似 Use this command to connect: ray start --address='192.168.1.100:6379' 的提示。
#### 步骤 2：在 Worker 节点启动

In [ ]:
ray start --address='192.168.1.100:6379'

- <font color='red'>解释：Worker 节点会根据地址主动连接 Head 节点的 GCS（全局控制服务）。</font>

### 场景三：资源隔离与限制

如果你不想让 Ray 占用机器上所有的 GPU（例如留一张卡给其他任务），可以手动指定资源。

命令：
- 解释：即使机器有 4 张显卡，Ray 也只会看到和使用前 3 张（CUDA:0, 1, 2）。

In [ ]:
ray start --head --num-gpus=3

## 🔍 启动后的进程结构

当你执行 ray start 后，系统会在后台启动一系列进程，主要包括：
1. GCS Server (仅 Head): <font color='red'>全局控制服务，存储集群元数据（如 Actor 位置、资源列表）。</font>
2. Raylet: 每个节点都有一个 Raylet 进程，包含两个核心组件：
    - <font color='red'>Node Manager:</font> 负责管理本节点的资源（CPU/GPU）和任务调度。
    - <font color='red'>Object Manager:</font> 负责节点间的数据传输（对象存储）。
3. Dashboard Agent: 负责收集本节点的监控数据并上报给 Head。
4. Workers: 实际执行 Python 代码的进程池。

## 🛑 停止 Ray
- 当你完成训练或开发后，可以使用以下命令<font color='red'>停止当前节点上的所有 Ray 进程：</font>
    - 注意：这会强制杀死所有与该 Ray 实例相关的进程。
    - <font color='red'>在多机环境下，需要在每台机器上分别执行，或者在 Head 节点使用 ray down（如果使用 Cluster Launcher）。</font>

In [ ]:
ray stop

## 💡 进阶：Ray 与 Torchrun 的关系

很多用户会混淆 ray start 和 torchrun。
- ray start: 是基础设施层。它负责把一堆机器组织成一个资源池（Cluster）。
- torchrun: 是应用层。它通常用于启动 PyTorch 的分布式训练（DDP）。

### 结合使用方式：
1. 在 Ray 集群中，你通常会先通过 ray start 把机器连起来，
2. 然后在 Ray 的 Python 代码（Driver）中，通过 subprocess 调用 torchrun 来启动具体的训练任务，或者直接使用 Ray 的原生 API（如 @ray.remote）来替代 torchrun 进行分布式计算。

# 在 Ray 中通过 subprocess 调用 torchrun，
- 通常用于“Ray 负责资源调度，PyTorch DDP 负责底层通信”的场景。


这是一个非常实用的需求。在 Ray 中通过 subprocess 调用 torchrun，通常用于“Ray 负责资源调度，PyTorch DDP 负责底层通信”的场景。
- 这样做的好处是：<font color='red'>你可以利用 Ray 的 Placement Group 来精确控制 GPU 的分配（例如确保 4 张卡在同一台机器上），然后让 torchrun 自动感知这些资源并启动分布式进程。</font>

以下是完整的代码实现，包含 Driver 脚本 和 训练脚本。
## 📂 文件结构
- <font color='red'>driver.py:</font> Ray 的入口脚本，负责申请资源并启动进程。
- <font color='red'>train.py:</font> 实际的 PyTorch 训练代码，通过 torchrun 启动。

1. 训练脚本 (train.py)
- 这是一个标准的 PyTorch DDP 脚本。注意，它不需要知道 Ray 的存在，只需要通过环境变量获取 torchrun 注入的信息（如 RANK, WORLD_SIZE）。

In [ ]:
import os
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.optim as optim
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, TensorDataset

def train():
    # 1. 初始化分布式环境
    # torchrun 会自动设置 RANK, WORLD_SIZE, MASTER_ADDR, MASTER_PORT 等环境变量
    dist.init_process_group(backend="nccl")
    
    # 获取当前进程的本地 Rank (即当前进程占用第几张卡)
    local_rank = int(os.environ["LOCAL_RANK"])
    
    rank = dist.get_rank()
    world_size = dist.get_world_size()

    # 2. 绑定 GPU
    # 这一步至关重要，确保每个进程只操作分配给它的卡
    torch.cuda.set_device(local_rank)
    device = torch.device(f"cuda:{local_rank}")

    print(f"🚀 Rank {rank}: 启动成功，使用设备 {device}")

    # 3. 构建模型并移动到 GPU
    model = nn.Linear(10, 10).to(device)
    
    model = DDP(model, device_ids=[local_rank])

    # 4. 构建数据 (模拟数据)
    # 实际场景中通常配合 DistributedSampler
    dataset = TensorDataset(torch.randn(100, 10).to(device), torch.randn(100, 10).to(device))
    # 注意：这里为了演示简单没有用 DistributedSampler，生产环境建议加上
    dataloader = DataLoader(dataset, batch_size=10)

    # 5. 模拟训练循环
    for step, (x, y) in enumerate(dataloader):
        model.zero_grad()
        loss = model(x).sum()
        loss.backward()
        
        # 模拟梯度同步
        dist.all_reduce(loss) 
        
        if rank == 0 and step % 2 == 0:
            print(f"Rank {rank}: Step {step}, Loss: {loss.item()}")

    # 6. 清理
    dist.destroy_process_group()
    print(f"✅ Rank {rank}: 训练结束")

if __name__ == "__main__":
    train()

2. Ray Driver 脚本 (driver.py)
- 这是核心部分。我们定义了一个 Ray Actor，并在其中使用 subprocess.Popen 来执行 torchrun 命令

In [ ]:
import ray
import os
import subprocess
import sys
import time
import socket

# --- 1. 定义 Ray Worker ---
# num_gpus=4 表示这个任务需要占用 4 张 GPU
@ray.remote(num_gpus=4)
class TorchrunWorker:
    def __init__(self, worker_id):
        self.worker_id = worker_id
        self.process = None

    def start(self, script_path, num_nodes, node_rank, master_addr, master_port):
        """
        构建并启动 torchrun 命令
        """
        # 获取 Ray 分配给当前 Actor 的 GPU ID (例如 [0, 1, 2, 3])
        # Ray 会自动设置 CUDA_VISIBLE_DEVICES 环境变量
        gpu_ids = ray.get_gpu_ids()
        num_gpus_per_node = len(gpu_ids)
        
        print(f"Worker {self.worker_id}: 准备启动 torchrun，分配到的 GPU: {gpu_ids}")

        # 构建 torchrun 命令
        # 注意：这里不需要指定 --nproc_per_node，torchrun 默认会使用所有可见 GPU
        # 但为了显式控制，我们通常设为 num_gpus_per_node
        cmd = [
            "torchrun",
            f"--nnodes={num_nodes}",
            f"--node_rank={node_rank}",
            f"--master_addr={master_addr}",
            f"--master_port={master_port}",
            f"--nproc_per_node={num_gpus_per_node}",
            script_path
        ]

        # 启动子进程
        # 这里不使用 shell=True，直接传递列表更安全
        self.process = subprocess.Popen(cmd)
        return f"Worker {self.worker_id} 已启动"

    def stop(self):
        if self.process:
            self.process.terminate()
            self.process.wait()
            print(f"Worker {self.worker_id} 已停止")

    def is_alive(self):
        return self.process is not None and self.process.poll() is None

# --- 2. Ray Driver 主逻辑 ---
def main():
    # 1. 连接或启动 Ray
    if not ray.is_initialized():
        ray.init()

    # 2. 确定 Master 地址 (通常取 Head 节点的 IP)
    # 在实际生产环境中，可能需要更复杂的逻辑来发现 Head IP
    master_addr = ray.util.get_node_ip_address()
    master_port = "29500" # 任意未被占用的端口

    # 3. 创建 Ray Actors
    # 假设我们要用 2 个节点，每个节点 4 张卡
    # 注意：在单机测试时，Ray 可能会把所有 Actor 调度到同一台机器，
    # 真正的多机需要 Ray 集群已经连接了多个物理节点
    num_nodes = 2 
    workers = []
    
    print(f"正在创建 {num_nodes} 个 Worker...")
    for i in range(num_nodes):
        # 创建 Actor
        worker = TorchrunWorker.remote(i)
        workers.append(worker)

    # 4. 并行启动所有 Worker
    print("正在启动分布式训练...")
    # 使用 ray.get 并发触发所有 worker 的 start 方法
    # 这里模拟了多机环境，实际上如果只有一台机器，node_rank 只是逻辑上的区分
    results = []
    for i, worker in enumerate(workers):
        # 传递参数：脚本路径，总节点数，当前节点Rank，主节点IP，主节点端口
        res = worker.start.remote("train.py", num_nodes, i, master_addr, master_port)
        results.append(res)
    
    # 等待启动指令发送完毕
    ray.get(results)
    print("所有节点训练任务已下发。")

    # 5. 监控状态
    try:
        while True:
            # 检查所有 worker 是否还在运行
            statuses = ray.get([w.is_alive.remote() for w in workers])
            if not all(statuses):
                print("检测到有节点退出，停止监控。")
                break
            time.sleep(5)
    except KeyboardInterrupt:
        print("\n收到中断信号，正在停止...")
        ray.get([w.stop.remote() for w in workers])

    ray.shutdown()

if __name__ == "__main__":
    main()

## 📝 代码详解与关键点
### 1. 为什么用 subprocess 而不是直接调用 Python 函数？
- 环境隔离：<font color='red'>torchrun 会启动多个子进程（每个 GPU 一个）。如果在 Ray Actor 的主线程中直接运行 Python 代码，很难管理这些子进程的生命周期（如 fork 问题）。使用 subprocess 可以将 PyTorch 的进程树与 Ray 的 Worker 进程隔离开。</font>

- 兼容性：很多现有的 PyTorch 项目都是设计为通过 torchrun 启动的。使用这种方式，你可以零修改（或极少修改）地复用现有的启动脚本。

### 2. CUDA_VISIBLE_DEVICES 的自动透传
- Ray 非常智能。当你定义 @ray.remote(num_gpus=4) 时，Ray 会在启动该 Actor 之前，自动设置环境变量 CUDA_VISIBLE_DEVICES（例如设置为 0,1,2,3）。
- <font color='red'>当你在 subprocess 中调用 torchrun 时，torchrun 会读取这个环境变量，自动知道它应该使用哪些 GPU，以及应该启动多少个进程（--nproc_per_node）。</font>
### 3. 多机通信 (master_addr)
- 在单机多卡测试时，master_addr 可以是 127.0.0.1 或 ray.util.get_node_ip_address()。
- <font color='red'>在真正的多机 Ray 集群中，所有 Worker 节点必须能够通过网络连接到 Head 节点（或指定的 Master 节点）。代码中使用 ray.util.get_node_ip_address() 通常能获取到当前物理节点的 IP，这是 Ray 推荐的做法。</font>

### 4. 进程生命周期管理
- 代码中使用了 Popen 而不是 call。这意味着 driver.py 不会阻塞，而是异步地启动任务。
- 通过 is_alive() 和 terminate()，Ray 可以优雅地控制训练任务的开始和结束。

## 🚀 如何运行
1. 保存文件：将上述代码分别保存为 driver.py 和 train.py。
2. 启动 Ray (可选，driver 中已包含初始化)：

In [ ]:
ray start --head --num-gpus=4

3. 运行 Driver：

In [ ]:
python driver.py

### 预期输出：
- 你会看到 Ray 调度资源，然后 train.py 被启动。由于我们模拟了 num_nodes=2，你会看到两组进程在运行（如果在单机上运行，它们会共享 4 张卡，逻辑上分为 Rank 0-3 和 Rank 4-7，具体取决于 torchrun 的配置，但在单机测试时建议将 num_nodes 设为 1 以避免网络握手超时）